# EX_00 — Introducción NLP

## Actividad 1

In [1]:
# Tareas típicas de HF que solemos usar
task_to_pipeline = {
    "sentiment-analysis": "sentiment-analysis",
    "ner": "ner",
    "summarization": "summarization"
}

for tarea, pipe in task_to_pipeline.items():
    print(f"Tarea: {tarea} -> Pipeline: {pipe}")

Tarea: sentiment-analysis -> Pipeline: sentiment-analysis
Tarea: ner -> Pipeline: ner
Tarea: summarization -> Pipeline: summarization


## Actividad 2

_Modelo elegido: distilbert-base-uncased-finetuned-sst-2-english_

* **Tarea:** Clasificación de texto (análisis de sentimiento positivo/negativo).
* **Datos de entrenamiento:** Se entrenó con el Stanford Sentiment Treebank (SST-2), que son frases de críticas de cine.
* **Limitaciones:** Al final está entrenado con datos de internet, así que puede tener sesgos. Además, este checkpoint en concreto solo funciona guay en inglés.


## Actividad 3

In [2]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

frase_buena = "I absolutely loved this practical AI course, it was incredibly useful!"
frase_rara = "The movie was long and had a lot of talking, but the ending was okay."

resultados = clasificador([frase_buena, frase_rara])

print("Resultado 1:", resultados[0])
print("Resultado 2:", resultados[1])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Resultado 1: {'label': 'POSITIVE', 'score': 0.9998083710670471}
Resultado 2: {'label': 'POSITIVE', 'score': 0.9991056323051453}


# EX_01 — Embeddings básicos

## Actividad 1

In [3]:
import numpy as np

vocab = ["cat", "sat", "mat", "the"]
sentence = ["the", "cat", "sat"]

idx_palabras = {palabra: i for i, palabra in enumerate(vocab)}
matriz_one_hot = np.zeros((len(sentence), len(vocab)))

for i, palabra in enumerate(sentence):
    if palabra in idx_palabras:
        matriz_one_hot[i, idx_palabras[palabra]] = 1

print("One-Hot Matrix:\n", matriz_one_hot)

One-Hot Matrix:
 [[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]


## Actividad 2

In [22]:
import numpy as np

def cosine_similarity(a, b):
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return np.dot(a, b) / (norma_a * norma_b)

vector_the = matriz_one_hot[0]
vector_cat = matriz_one_hot[1]

print("Similitud:", cosine_similarity(vector_the, vector_cat))

# Porque todas las palabras son vectores ortogonales.
# "perro" y "gato" son tan distintos entre sí como "perro" y "televisor"

Similitud: 0.0


## Actividad 3

In [5]:
from gensim.models import Word2Vec

sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "log"],
]

modelo_w2v = Word2Vec(sentences, vector_size=8, window=2, min_count=1)
print("Similitud entre 'cat' y 'dog':", modelo_w2v.wv.similarity("cat", "dog"))

Similitud entre 'cat' y 'dog': -0.09075314


# EX_02 — Embeddings con Transformers

## Actividad 1

In [6]:
from transformers import AutoTokenizer, AutoModel
import torch

text = "Transformers build contextual embeddings."
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.last_hidden_state
mask = inputs['attention_mask'].unsqueeze(-1).expand(hidden_states.size()).float()

suma_emb = torch.sum(hidden_states * mask, dim=1)
suma_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
mean_pooled = suma_emb / suma_mask

print("Shape después del pooling:", mean_pooled.shape)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\Antonio\anaconda3\envs\UEA_IA\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Antonio\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape después del pooling: torch.Size([1, 768])


## Actividad 2

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
from numpy.linalg import norm

modelo_st = SentenceTransformer('all-MiniLM-L6-v2')

frases = ["I love machine learning.", "Deep learning is my passion."]
embs = modelo_st.encode(frases)

similitud = np.dot(embs[0], embs[1]) / (norm(embs[0]) * norm(embs[1]))
print("Similitud coseno:", similitud)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similitud coseno: 0.6496212


## Actividad 3

In [8]:
docs = [
    "Las acciones de la bolsa han caído muchísimo hoy.",
    "El mercado financiero se ha desplomado esta mañana.",
    "Para hacer esta tarta necesitas tres huevos."
]

vectores = modelo_st.encode(docs)

def calc_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Similitud A vs B (Paráfrasis):", calc_sim(vectores[0], vectores[1]))
print("Similitud A vs C (Nada que ver):", calc_sim(vectores[0], vectores[2]))

Similitud A vs B (Paráfrasis): 0.4441549
Similitud A vs C (Nada que ver): 0.43807417


# EX_03 — Ingeniería de prompts

## Actividad 1

In [9]:
system_prompt = '''Eres un asistente de datos super preciso. Cuando el usuario te haga una pregunta de cultura general, tienes que devolver SOLO un objeto JSON válido.'''

user_prompt = '¿Cuál es la capital de Australia y en qué año se fundó?'

## Actividad 2

In [10]:
few_shot_template = '''
Vas a clasificar el sentimiento de las reseñas de un restaurante. Opciones: positive, negative o neutral.

Reseña: "La comida estaba fría y tardaron una hora."
Sentimiento: negative

Reseña: "El menú del día tiene arroz y pescado."
Sentimiento: neutral

Reseña: "{user_input}"
Sentimiento:
'''
print(few_shot_template)


Vas a clasificar el sentimiento de las reseñas de un restaurante. Opciones: positive, negative o neutral.

Reseña: "La comida estaba fría y tardaron una hora."
Sentimiento: negative

Reseña: "El menú del día tiene arroz y pescado."
Sentimiento: neutral

Reseña: "{user_input}"
Sentimiento:



## Actividad 3

In [11]:
step1 = 'Contesta a la siguiente duda técnica del usuario: {question}'

step2 = '''Evalúa la respuesta generada por una IA.
Respuesta a evaluar: {ai_response}
Dime qué falla y proponme una versión mejor.'''

# EX_04 — Chatbots básicos

## Actividad 1

In [12]:
messages = [
    {"role": "system", "content": "Eres un profe particular de programación usando el método socrático."},
    {"role": "user", "content": "Me sale un IndexError cuando intento recorrer esta lista en Python, ¿por qué?"}
]

## Actividad 2

In [13]:
from typing import Any

def trim_history(messages: list[dict[str, Any]], max_turns: int) -> list[dict[str, Any]]:
    if not messages:
        return []
    sys_prompt = [m for m in messages if m.get("role") == "system"]
    resto = [m for m in messages if m.get("role") != "system"]
    mensajes_a_guardar = max_turns * 2
    resto_recortado = resto[-mensajes_a_guardar:] if mensajes_a_guardar > 0 else []
    return sys_prompt + resto_recortado

## Actividad 3

In [14]:
summarize_prompt = '''
Hazme un resumen súper conciso de lo que hemos hablado hasta ahora.

Historial de chat:
{chat_history}

Resumen:
'''

# EX_05 — Vectorstores y Retrieval

## Actividad 1

In [15]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks = []
    paso = chunk_size - overlap
    for i in range(0, len(text), paso):
        chunks.append(text[i:i + chunk_size])
    return chunks

texto_largo = "bla " * 600
mis_chunks = chunk_text(texto_largo)

## Actividad 2

In [16]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer('all-MiniLM-L6-v2')
textos = ["El cielo es azul.", "El coche es rápido.", "La IA avanza a lo bestia."]

vectores = modelo.encode(textos)
faiss.normalize_L2(vectores)

dimension = vectores.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(vectores)

query = "¿A qué velocidad va el vehículo?"
vec_query = modelo.encode([query])
faiss.normalize_L2(vec_query)

distancias, indices = index.search(vec_query, 2)
print("Top 2 recuperados:")
for idx in indices[0]:
    print("-", textos[idx])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top 2 recuperados:
- La IA avanza a lo bestia.
- El cielo es azul.


## Actividad 3

In [17]:
from numpy.linalg import norm

modelo = SentenceTransformer('all-MiniLM-L6-v2')

pregunta = "¿Cuáles son las ventajas de las bases de datos vectoriales?"
documentos = [
    "Las naranjas tienen mucha vitamina C.",
    "Una base de datos vectorial permite hacer búsquedas por similitud semántica.",
    "El partido acabó empate a cero.",
]

embs = modelo.encode([pregunta] + documentos)
q_emb = embs[0]

for i, d_emb in enumerate(embs[1:]):
    score = np.dot(q_emb, d_emb) / (norm(q_emb) * norm(d_emb))
    print(score)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.25840092
0.59949386
0.28437498


# EX_06 — Introducción a RAG

## Actividad 1

In [18]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    textos_contexto = "\n".join([f"- {c}" for c in context_chunks])
    prompt = f'''Usa el siguiente contexto para responder a la pregunta del usuario.

### Context
{textos_contexto}

### Question
{question}
'''
    return prompt

## Actividad 2

In [19]:
mis_chunks = [
    "RAG significa Retrieval-Augmented Generation.",
    "Se usa para que los LLM no alucinen pasándoles documentos."
]
duda = "¿Qué es eso de RAG?"

print(build_prompt(mis_chunks, duda))

Usa el siguiente contexto para responder a la pregunta del usuario.

### Context
- RAG significa Retrieval-Augmented Generation.
- Se usa para que los LLM no alucinen pasándoles documentos.

### Question
¿Qué es eso de RAG?



## Actividad 3

_Tu explicación:_

1. Poner un umbral al Score de similitud.
2. Forzar al LLM en el Prompt para responder solo con contexto relevante.


# EX_07 — Reranking y optimización

## Actividad 1

In [20]:
import numpy as np

query = "latency vs throughput"
docs = [f"doc{i}: ..." for i in range(5)]

bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3])

top4_idx = np.argsort(bi_scores)[::-1][:4]
scores_filtrados = cross_scores[top4_idx]
orden_local = np.argsort(scores_filtrados)[::-1]
indices_finales = top4_idx[orden_local]

for i in indices_finales:
    print(i)

1
3
4
0


## Actividad 2

In [23]:
# Pseudocódigo de cómo funciona MMR (Maximal Marginal Relevance)
# 
# def calcular_mmr(documentos, query_emb, k=3, lambda_val=0.5):
#     seleccionados = []
#     candidatos = documentos.copy()
#     
#     while len(seleccionados) < k:
#         mejor_doc = None
#         mejor_score = -9999
#         
#         for doc in candidatos:
#             # Qué tan relevante es para la pregunta
#             relevancia = sim(query_emb, doc.emb)
#             
#             # Qué tan parecido es a lo que YA hemos seleccionado (queremos penalizar esto)
#             diversidad = max([sim(doc.emb, s.emb) for s in seleccionados]) if seleccionados else 0
#             
#             # La fórmula mágica: balanceamos relevancia vs repetición
#             score_mmr = (lambda_val * relevancia) - ((1 - lambda_val) * diversidad)
#             
#             if score_mmr > mejor_score:
#                 mejor_score = score_mmr
#                 mejor_doc = doc
#                 
#         seleccionados.append(mejor_doc)
#         candidatos.remove(mejor_doc)
#         
#     return seleccionados

## Actividad 3

| Etapa | Coste relativo (1-10) | Por qué |
|---|---:|---|
| Embedding query | 1 | Muy barato |
| Cross-encoder | 4 | Más pesado |
| Generación LLM | 10 | Lo más costoso |
